In [3]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

import joblib


In [4]:
def process_iwris_folder(folder, output_col, agg="mean"):
    all_rows = []

    for file in os.listdir(folder):
        if not file.endswith(".xlsx"):
            continue

        path = os.path.join(folder, file)
        xls = pd.ExcelFile(path)

        # Always second sheet = data
        sheet = xls.sheet_names[1]

        df = pd.read_excel(path, sheet_name=sheet, skiprows=6)
        df.columns = df.columns.astype(str).str.strip()

        #  Auto-detect datetime column
        date_col = None
        for col in df.columns:
            if "date" in col.lower() or "time" in col.lower():
                date_col = col
                break

        if date_col is None:
            raise ValueError(f"No datetime column found in {file}")

        #  Auto-detect value column (last numeric-like column)
        value_col = None
        for col in reversed(df.columns):
            if pd.to_numeric(df[col], errors="coerce").notna().sum() > 0:
                value_col = col
                break

        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

        df = df.dropna(subset=[date_col, value_col])

        df["Year"] = df[date_col].dt.year
        df["Month"] = df[date_col].dt.month

        if agg == "sum":
            monthly = df.groupby(["Year", "Month"])[value_col].sum().reset_index()
        else:
            monthly = df.groupby(["Year", "Month"])[value_col].mean().reset_index()

        monthly = monthly.rename(columns={value_col: output_col})
        all_rows.append(monthly)

    return pd.concat(all_rows, ignore_index=True)


In [5]:
rain_monthly = process_iwris_folder(
    "iwris_rainfall", "Rainfall", agg="sum"
)

discharge_monthly = process_iwris_folder(
    "iwris_discharge", "Discharge", agg="mean"
)

waterlevel_monthly = process_iwris_folder(
    "iwris_waterlevel", "Water_Level", agg="mean"
)


In [6]:
iwris_monthly = (
    rain_monthly
    .merge(discharge_monthly, on=["Year", "Month"], how="inner")
    .merge(waterlevel_monthly, on=["Year", "Month"], how="inner")
)

iwris_monthly.head()


,Year,Month,Rainfall,Discharge,Water_Level
0,2020,1,24.0,0.0,0.000000
1,2020,1,24.0,0.0,185.666452
2,2020,1,24.0,0.0,25.549999
3,2020,1,24.0,0.0,29.000000
4,2020,1,24.0,0.0,81.497922


In [7]:
def load_cpcb_year(file_path, year):
    # Read Table 2 (actual data table)
    df = pd.read_excel(
        file_path,
        sheet_name="Table 2",
        skiprows=5,
        header=None
    )

    # Replace CPCB non-numeric markers
    df = df.replace(
        ["BDL", "bdl", "-", "NA", "N.A.", ""],
        pd.NA
    )

    # Convert required columns to numeric BEFORE mean
    for col in [5, 6, 7, 8, 10]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Now safely compute
    DO = df[[5, 6]].mean(axis=1)
    pH = df[[7, 8]].mean(axis=1)
    BOD = df[10]

    # Aggregate for the year
    result = pd.DataFrame({
        "Year": [year],
        "Month": [1],          # CPCB annual → January
        "pH": [pH.mean()],
        "DO": [DO.mean()],
        "BOD": [BOD.mean()]
    })

    return result



In [8]:
cpcb_files = {
    2020: "WQuality_River-Data-2020.xlsx",
    2021: "WQuality_River-Data-2021.xlsx",
    2022: "WQuality_River-Data-2022.xlsx",
    2023: "WQuality_River-Data-2023.xlsx",
}

dfs = []
for year, file in cpcb_files.items():
    dfs.append(load_cpcb_year(file, year))

cpcb_monthly = pd.concat(dfs, ignore_index=True)
cpcb_monthly


,Year,Month,pH,DO,BOD
0,2020,1,7.778182,8.648182,1315.381818
1,2021,1,8.433333,13.233333,71.666667
2,2022,1,7.895441,8.442647,255.647059
3,2023,1,7.886364,8.268182,446.500000


In [14]:
# ================================
# STEP 1: Aggregate IWRS monthly
# ================================

iwris_agg = (
    iwris_monthly
    .groupby(["Year", "Month"], as_index=False)
    .agg({
        "Rainfall": "mean",
        "Discharge": "mean",
        "Water_Level": "mean"
    })
)

print("IWRS aggregated shape:", iwris_agg.shape)
iwris_agg.head()
print(cpcb_monthly.shape)
cpcb_monthly.head()
# ================================
# STEP 3: Merge IWRS + CPCB
# ================================

final_df = pd.merge(
    iwris_agg,
    cpcb_monthly,
    on=["Year", "Month"],
    how="inner"   # IMPORTANT
)

final_df = final_df.dropna().reset_index(drop=True)

print("Final dataset shape:", final_df.shape)
final_df.head()


IWRS aggregated shape: (25, 5)
(4, 5)
Final dataset shape: (3, 8)


,Year,Month,Rainfall,Discharge,Water_Level,pH,DO,BOD
0,2020,1,5.680000,15.968189,54.258899,7.778182,8.648182,1315.381818
1,2021,1,30.164445,16.300163,28.463937,8.433333,13.233333,71.666667
2,2022,1,0.363636,10.066000,35.743375,7.895441,8.442647,255.647059


In [17]:

cpcb_expanded = []

for _, row in cpcb_monthly.iterrows():
    for m in range(1, 13):
        cpcb_expanded.append({
            "Year": row["Year"],
            "Month": m,
            "pH": row["pH"],
            "DO": row["DO"],
            "BOD": row["BOD"]
        })

cpcb_monthly_expanded = pd.DataFrame(cpcb_expanded)

print("Expanded CPCB shape:", cpcb_monthly_expanded.shape)
cpcb_monthly_expanded.head()



Expanded CPCB shape: (48, 5)


,Year,Month,pH,DO,BOD
0,2020.0,1,7.778182,8.648182,1315.381818
1,2020.0,2,7.778182,8.648182,1315.381818
2,2020.0,3,7.778182,8.648182,1315.381818
3,2020.0,4,7.778182,8.648182,1315.381818
4,2020.0,5,7.778182,8.648182,1315.381818


In [18]:


final_df = iwris_monthly.merge(
    cpcb_monthly_expanded,
    on=["Year", "Month"],
    how="inner"
)

final_df = final_df.dropna().reset_index(drop=True)

print("Final dataset shape:", final_df.shape)
final_df.head()



Final dataset shape: (11925, 8)


,Year,Month,Rainfall,Discharge,Water_Level,pH,DO,BOD
0,2020,1,24.0,0.0,0.000000,7.778182,8.648182,1315.381818
1,2020,1,24.0,0.0,185.666452,7.778182,8.648182,1315.381818
2,2020,1,24.0,0.0,25.549999,7.778182,8.648182,1315.381818
3,2020,1,24.0,0.0,29.000000,7.778182,8.648182,1315.381818
4,2020,1,24.0,0.0,81.497922,7.778182,8.648182,1315.381818


In [22]:
final_df = final_df.copy()

final_df["BOD_rank"] = final_df["BOD"].rank(pct=True)

def pollution_label(row):
    if row["BOD_rank"] <= 0.33:
        return "Clean"
    elif row["BOD_rank"] <= 0.66:
        return "Moderate"
    else:
        return "Polluted"

final_df["Pollution_Level"] = final_df.apply(pollution_label, axis=1)
final_df.drop(columns="BOD_rank", inplace=True)

print(final_df["Pollution_Level"].value_counts())




Pollution_Level
Polluted    6132
Clean       5441
Moderate     352
Name: count, dtype: int64


In [25]:
X = final_df[["Rainfall", "Discharge", "Water_Level"]]
y = final_df["Pollution_Level"]

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split (IMPORTANT: stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluation
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

       Clean       0.89      0.81      0.85      1088
    Moderate       1.00      0.52      0.69        71
    Polluted       0.83      0.92      0.87      1226

    accuracy                           0.86      2385
   macro avg       0.91      0.75      0.80      2385
weighted avg       0.86      0.86      0.86      2385



In [26]:
import joblib

# Save trained model
joblib.dump(model, "riversight_numeric_model.pkl")

# Save label encoder
joblib.dump(le, "pollution_label_encoder.pkl")

print(" Model saved as riversight_numeric_model.pkl")
print(" Label encoder saved as pollution_label_encoder.pkl")


 Model saved as riversight_numeric_model.pkl
 Label encoder saved as pollution_label_encoder.pkl
